<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Solution: A forecast is a range

**[A forecast is a range](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/)** · Forecasting and Time Series: Predict What Comes Next · Module 7, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** make a forecast with a prediction interval (a range), read its quantiles, draw a fan chart, and count how often the real values fall inside the range.

| | |
|---|---|
| **Time** | About 35 minutes, of which about 1 minute of waiting for the backtest |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | ETS, from [Exponential smoothing](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/). Quantiles, from the [mathematics course](https://learning.nextia-ai.com/courses/math/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M07-L01-a-forecast-is-a-range/a-forecast-is-a-range-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Five days, one range

A forecast says 20 units a day, and its **80% prediction interval** goes from 14 to 26. This means: if the model is right about its own errors, the real value falls inside the range on about 8 days out of 10. The low end is the **10% quantile** (q10): the real value is below it 10% of the time. The high end is the **90% quantile** (q90).

> **Predict.** The real values are 18, 27, 21, 13 and 24. How many fall inside 14 to 26? Then run the cell.

In [ ]:
real = np.array([18, 27, 21, 13, 24])
q10, q90 = 14, 26
inside = (real >= q10) & (real <= q90)
print(inside.tolist(), "->", int(inside.sum()), "of", len(real), "inside")

> **Check.** You should see `[True, False, True, False, True] -> 3 of 5 inside`.

Five days are far too few to judge a range. With many days, the share inside is the **coverage**: for an honest 80% range, it should be close to 80%.

## 2. The parasol's range before the heat wave

statsforecast gives a range when you pass `level=[80, 90]`: the columns `AutoETS-lo-80`, `AutoETS-hi-80` and so on. The next cell forecasts the parasol for 14 days from 22 June 2025 and draws a **fan chart**: the 80% range inside the 90% range.

> **Predict.** The heat wave starts on 30 June. Will the real sales stay inside the 90% range? Then run the cell.

In [ ]:
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

parasol = load_sales(["P025"])
origin = pd.Timestamp("2025-06-22")
train = parasol[parasol["ds"] <= origin][["unique_id", "ds", "y"]]
real = parasol[(parasol["ds"] > origin) & (parasol["ds"] <= origin + pd.Timedelta(days=14))]
fc = StatsForecast(models=[AutoETS(season_length=7)], freq="D").forecast(df=train, h=14, level=[80, 90])
fc = fc.merge(real[["ds", "y"]], on="ds")
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train["ds"].tail(35), train["y"].tail(35), color="grey", label="history")
ax.fill_between(fc["ds"], fc["AutoETS-lo-90"].clip(lower=0), fc["AutoETS-hi-90"], alpha=0.2, color="tab:blue", label="90% range")
ax.fill_between(fc["ds"], fc["AutoETS-lo-80"].clip(lower=0), fc["AutoETS-hi-80"], alpha=0.3, color="tab:blue", label="80% range")
ax.plot(fc["ds"], fc["AutoETS"], color="tab:blue", label="forecast")
ax.plot(fc["ds"], fc["y"], "k.-", label="real sales")
ax.set_xlabel("Day")
ax.set_ylabel("Parasols sold")
ax.legend(loc="upper left")
plt.show()
above = fc[fc["y"] > fc["AutoETS-hi-90"]]
print("days above the 90% range:", [d.strftime("%a %d %b") for d in above["ds"]])

> **Check.** You should see a band that widens a little with each day ahead, and `days above the 90% range: ['Sun 06 Jul']`.

Only one day of the heat wave is above the 90% range: the parasol is a noisy product, so its range is wide. The forecast in the middle is still far too low all week. A 90% range is wrong about 1 day in 10, and the days it misses are often the ones that matter most.

## 3. Coverage over the course backtest

The next cell runs ETS with ranges at the 13 cut-offs for the 50 products, and counts the share of real values inside the 80% range.

In [ ]:
def ets_range(train, h, future):
    f = StatsForecast(models=[AutoETS(season_length=7)], freq="D").forecast(
        df=train[["unique_id", "ds", "y"]], h=h, level=[80, 90])
    out = pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f["AutoETS"],
                        "q5": f["AutoETS-lo-90"], "q10": f["AutoETS-lo-80"], "q50": f["AutoETS"],
                        "q90": f["AutoETS-hi-80"], "q95": f["AutoETS-hi-90"]})
    return out.assign(**{c: out[c].clip(lower=0) for c in ["yhat", "q5", "q10", "q50", "q90", "q95"]})

sales = load_sales(BACKTEST_PRODUCTS)
fc_range = bt.backtest(sales, ets_range, h=14, origins=ORIGINS)
print("coverage of the 80% range:", bt.coverage(fc_range, "q10", "q90"))
row = bt.score(fc_range, sales, "ETS with ranges", season=7, cost=bt.stock_cost(PRICES))
print("pinball (the score of the quantiles, Module 7 lesson 2):", row["pinball"])

> **Check.** You should see `coverage of the 80% range: 0.835` and a pinball value of 0.11.

The ETS ranges hold a little more than they promise: 83.5% of the real values fall inside the 80% range. A range that is too wide is not free either: it makes Kenji order too much. Module 7, lesson 2 scores ranges with the pinball loss.

> **Your turn.** Compute the coverage of the **90%** range (the columns `q5` and `q95`) with `bt.coverage`. Put it in `coverage_90`. Then run the check cell.

In [ ]:
coverage_90 = bt.coverage(fc_range, "q5", "q95")
print(coverage_90)

In [ ]:
expect('coverage_90', coverage_90, 0.905)

## 4. Failure case: ordering the middle of the range

Kenji orders the forecast added up over 14 days: the middle of the range. The next cell counts how often that order was smaller than the real sales of the 14 days (a stock-out somewhere in the period).

In [ ]:
g = fc_range.groupby(["unique_id", "origin"]).agg(sold=("y", "sum"), order=("yhat", "sum"))
short = (g["sold"] > np.ceil(g["order"])).mean()
print(f"orders too small: {short:.0%} of {len(g)} orders")

> **Check.** You should see `orders too small: 50% of 650 orders`.

An order at the middle of the range runs short about half of the time, whatever the model. The point forecast cannot tell Kenji how much more to order to be safe 90% of the time, or how much less when a left-over item costs more than a lost sale. **The range can.** Module 7, lesson 3 chooses the order from the range and the costs.

## Next

You can now make a range, draw it and count its coverage. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Score the range](https://learning.nextia-ai.com/courses/forecasting/m07/score-the-range/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m07/a-forecast-is-a-range/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.